# Model Evaluation: Predicting Conflict Escalation (Target k=1.75)

* **Model A:** Baseline (Tabular ACLED + Food + Rain)
* **Model B (Conflict-Only):** Baseline + Conflict-Only Text Embeddings (PCA)
* **Model B (All-Event):** Baseline + All-Event Text Embeddings (Non-PCA)

*Key finding: Incorporating the narrative context of ALL events (violent and non-violent) improves the model's ability to balance early-warning recall with precision.*

In [28]:
import pandas as pd
import plotly.express as px

In [ ]:
model_a_results = pd.read_json(
    "evaluation/model_reports/Model_A_results.json", typ="series", convert_dates=False
).to_frame(name="Value")
model_a_results["model"] = "Model A"
model_b_conflict_results = pd.read_json(
    "evaluation/model_reports/Model_B_conflict-only_text_results.json",
    typ="series",
    convert_dates=False,
).to_frame(name="Value")
model_b_conflict_results["model"] = "Model B - conflict"
model_b_all_results = pd.read_json(
    "evaluation/model_reports/Model_B_all-event_text_results.json",
    typ="series",
    convert_dates=False,
).to_frame(name="Value")
model_b_all_results["model"] = "Model B - all"

all_results = pd.concat(
    [model_a_results, model_b_all_results, model_b_conflict_results]
)

In [36]:
df_plot = all_results.reset_index()
df_plot.columns = ['Metric', 'Score', 'Model']

metrics_to_plot = [
    'onset_f1_class1', 
    'onset_recall_class1', 
    'onset_precision_class1', 
    'onset_aupr'
]
df_plot = df_plot[df_plot['Metric'].isin(metrics_to_plot)].copy()

metric_mapping = {
    'onset_f1_class1': 'Onset F1 Score',
    'onset_recall_class1': 'Onset Recall',
    'onset_precision_class1': 'Onset Precision',
    'onset_aupr': 'Onset AUPR'
}
df_plot['Metric'] = df_plot['Metric'].map(metric_mapping)

color_discrete_map = {
    "Model A": "#6c757d",              # Grey for Baseline
    "Model B - conflict": "#dc3545",   # Red for Conflict-Only (Update this string if it's named differently in your df!)
    "Model B - all": "#198754"         # Green for All-Event
}


fig = px.bar(
    df_plot, 
    x="Metric", 
    y="Score", 
    color="Model", 
    barmode="group",
    text_auto='.4f', 
    color_discrete_map=color_discrete_map,
    title="Performance Comparison: Onset"
)

fig.update_layout(
    xaxis_title="",
    yaxis_title="Score (0.0 to 1.0)",
    legend_title_text="",
    yaxis=dict(range=[0, 1]), # Keeps the scale locked so differences are obvious
    title_font_size=20,
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_traces(textposition='outside')

fig.show()

In [37]:
df_plot = all_results.reset_index()
df_plot.columns = ['Metric', 'Score', 'Model']

metrics_to_plot = [
    'active_f1_class1', 
    'active_recall_class1', 
    'active_precision_class1', 
    'active_aupr'
]
df_plot = df_plot[df_plot['Metric'].isin(metrics_to_plot)].copy()

metric_mapping = {
    'active_f1_class1': 'Active F1 Score',
    'active_recall_class1': 'Active Recall',
    'active_precision_class1': 'Active Precision',
    'active_aupr': 'Active AUPR'
}
df_plot['Metric'] = df_plot['Metric'].map(metric_mapping)

color_discrete_map = {
    "Model A": "#6c757d",              # Grey for Baseline
    "Model B - conflict": "#dc3545",   # Red for Conflict-Only (Update this string if it's named differently in your df!)
    "Model B - all": "#198754"         # Green for All-Event
}


fig = px.bar(
    df_plot, 
    x="Metric", 
    y="Score", 
    color="Model", 
    barmode="group",
    text_auto='.4f', 
    color_discrete_map=color_discrete_map,
    title="Performance Comparison: Active"
)

fig.update_layout(
    xaxis_title="",
    yaxis_title="Score (0.0 to 1.0)",
    legend_title_text="",
    yaxis=dict(range=[0, 1]), # Keeps the scale locked so differences are obvious
    title_font_size=20,
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_traces(textposition='outside')

fig.show()

# Onset vs active war

The test data was split across two periods: the onset of war and active civil war. As expected, models perform differently across these two distinct conflict scenarios.

When trying to detect the possible onset of war, richer contextual information is vital (such as all text, non-PCA). Pre the onset of war, many months may go by without any recorded conflict events, text embeddings add non-zero precursor signals. 

However, when conflict has been ongoing for multiple months, the statistical baseline becomes the superior performer. Region-month counts are no longer zero-inflated and text becomes noise.

In [42]:
shap_all = pd.read_csv('evaluation/model_reports/Model_B_all-event_text_shap.csv')
shap_conflict = pd.read_csv('evaluation/model_reports/Model_B_conflict-only_text_shap.csv')
shap_a = pd.read_csv('evaluation/model_reports/Model_A_shap.csv')

def categorize_feature(feat_name):
    feat_name = str(feat_name).lower()
    if feat_name.startswith('emb_') or feat_name.startswith('pc'):
        return 'Text Embeddings'
    elif feat_name.startswith('rolling_'):
        return 'Structural Baseline (Rolling Stats)'
    elif 'rain' in feat_name:
        return 'Rainfall'
    elif 'price' in feat_name:
        return 'Food Prices'
    else:
        return 'Tabular ACLED Counts'

shap_all['Category'] = shap_all['feature'].apply(categorize_feature)
shap_conflict['Category'] = shap_conflict['feature'].apply(categorize_feature)
shap_a['Category'] = shap_a['feature'].apply(categorize_feature)


sum_all = shap_all.groupby('Category')['mean_abs_shap'].sum().reset_index()
sum_all['Model'] = 'Model B (All-Event Text)'

sum_conflict = shap_conflict.groupby('Category')['mean_abs_shap'].sum().reset_index()
sum_conflict['Model'] = 'Model B (Conflict-Only Text)'

sum_a = shap_a.groupby('Category')['mean_abs_shap'].sum().reset_index()
sum_a['Model'] = 'Model A (Baseline)'

df_combined = pd.concat([sum_a, sum_conflict, sum_all])

df_combined['Total_SHAP'] = df_combined.groupby('Model')['mean_abs_shap'].transform('sum')
df_combined['% Importance'] = (df_combined['mean_abs_shap'] / df_combined['Total_SHAP']) * 100


color_map = {
    'Text Embeddings': '#0d6efd',                 # Blue
    'Structural Baseline (Rolling Stats)': '#6c757d', # Grey
    'Tabular ACLED Counts': '#ffc107',            # Yellow
    'Food Prices': '#198754',                     # Green
    'Climate': '#0dcaf0'                          # Light Blue
}


fig = px.bar(
    df_combined, 
    x="Model", 
    y="% Importance", 
    color="Category", 
    color_discrete_map=color_map,
    title="SHAP Feature Importance (top 30 features)",
    text_auto='.1f'
)

fig.update_layout(
    xaxis_title="",
    yaxis_title="Relative importance (%)",
    legend_title_text="Feature Category",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()